# Hadoop Streaming — Custom MapReduce in Your Own Code

Lab 05 ran a **prebuilt** WordCount jar. Real analysis needs **your own logic**. *Hadoop Streaming* lets any executable that reads stdin and writes stdout act as a mapper/reducer — so we write the logic as a tiny script and still get full distributed execution on YARN.

**The classic Hadoop example:** *maximum temperature per year* from weather records (Tom White, *Hadoop: The Definitive Guide*). We use real, large, public data.

**Dataset:** [NOAA GHCN‑Daily](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily), one gzipped CSV per year — no authentication. Each recent year is **~1 GB uncompressed**, so even one or two years is a genuine big-data workload.

> Hadoop reads `.gz` input transparently, so we upload the compressed files straight to HDFS.

## Setup

In [ ]:
import subprocess
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to HDFS via proxy ✓')


def hadoop(cmd: str, container: str = 'namenode', timeout: int = 1800) -> str:
    """Run a command inside a cluster container (yarn/hdfs ops)."""
    try:
        p = subprocess.run(['docker', 'exec', container, 'bash', '-lc', cmd],
                           capture_output=True, text=True, timeout=timeout)
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        print(f'⚠️  Could not run docker exec ({e}).')
        print(f'    Run manually inside `make shell-{container}`:\n    {cmd}')
        return ''
    if p.returncode != 0:
        print(f'⚠️  Exit {p.returncode}. stderr tail:\n{p.stderr[-2000:]}')
    return p.stdout + p.stderr


def docker_cp(local: str, dest: str, container: str = 'namenode'):
    """Copy a local file into a container."""
    subprocess.run(['docker', 'exec', container, 'mkdir', '-p',
                    dest.rsplit('/', 1)[0]], capture_output=True)
    subprocess.run(['docker', 'cp', local, f'{container}:{dest}'], check=True)

## 1. Download NOAA weather years

**Start with one year**; add more to `YEARS` for a multi‑GB run.

In [ ]:
import os
import requests

os.makedirs('../temp/weather', exist_ok=True)

YEARS = [2023]          # ↑ e.g. [2021, 2022, 2023] for ~3 GB uncompressed
BASE = 'https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/by_year/{}.csv.gz'


def download_stream(url, dest, chunk=1024 * 1024):
    if os.path.exists(dest):
        print(f'  cached: {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)')
        return
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest, 'wb') as fh:
            for part in r.iter_content(chunk_size=chunk):
                fh.write(part)
    print(f'  downloaded: {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)')


local_years = []
for y in YEARS:
    dest = f'../temp/weather/{y}.csv.gz'
    download_stream(BASE.format(y), dest)
    local_years.append(dest)

## 2. Ingest into HDFS

Each GHCN row is: `STATION,YYYYMMDD,ELEMENT,VALUE,M-FLAG,Q-FLAG,S-FLAG,OBS-TIME`. `TMAX` is the daily maximum temperature in **tenths of °C**.

In [ ]:
INPUT_DIR = '/datasets/weather'
client.makedirs(INPUT_DIR, permission=0o755)

for f in local_years:
    client.upload(f'{INPUT_DIR}/{os.path.basename(f)}', f, overwrite=True)

print('In HDFS:', client.list(INPUT_DIR))

## 3. Write the mapper and reducer

**Mapper** — emit `year \t TMAX` for valid maximum-temperature rows.
**Reducer** — for each year (keys arrive sorted), compute max / mean / count.

We write Python versions (readable) and, in case the cluster image lacks a Python interpreter, equivalent `awk` versions as a fallback.

In [ ]:
mapper_py = r'''import sys
for line in sys.stdin:
    f = line.split(',')
    if len(f) > 3 and f[2] == 'TMAX' and f[3]:
        try:
            print('%s\t%d' % (f[1][:4], int(f[3])))
        except ValueError:
            pass
'''

reducer_py = r'''import sys
cur = None; cnt = 0; tot = 0; mx = None
def flush(y, cnt, tot, mx):
    if cnt:
        print('%s\t%.1f\t%.1f\t%d' % (y, mx/10.0, tot/cnt/10.0, cnt))
for line in sys.stdin:
    try:
        y, v = line.rstrip('\n').split('\t'); v = int(v)
    except ValueError:
        continue
    if y != cur:
        if cur is not None: flush(cur, cnt, tot, mx)
        cur = y; cnt = 0; tot = 0; mx = v
    cnt += 1; tot += v
    if v > mx: mx = v
if cur is not None: flush(cur, cnt, tot, mx)
'''

# awk fallbacks (TMAX in tenths °C; reducer divides by 10)
mapper_awk = r'''#!/usr/bin/awk -f
BEGIN { FS = "," }
$3 == "TMAX" && $4 != "" { print substr($2, 1, 4) "\t" $4 }
'''

reducer_awk = r'''#!/usr/bin/awk -f
BEGIN { FS = "\t"; cur = "" }
{
  if ($1 != cur) {
    if (cur != "") printf "%s\t%.1f\t%.1f\t%d\n", cur, mx/10, tot/cnt/10, cnt
    cur = $1; cnt = 0; tot = 0; mx = -99999
  }
  cnt++; tot += $2; if ($2 > mx) mx = $2
}
END { if (cur != "") printf "%s\t%.1f\t%.1f\t%d\n", cur, mx/10, tot/cnt/10, cnt }
'''

for name, content in [('mapper.py', mapper_py), ('reducer.py', reducer_py),
                      ('mapper.awk', mapper_awk), ('reducer.awk', reducer_awk)]:
    with open(f'../temp/weather/{name}', 'w') as fh:
        fh.write(content)
print('Wrote mapper/reducer scripts to ../temp/weather/')

## 4. Pick an interpreter & ship the scripts into the cluster

Streaming tasks run on the NodeManager. We detect whether the image has `python3`; if not, we fall back to `awk` (always present). The chosen scripts are copied into the NameNode container, from where we submit the job.

In [ ]:
has_python = bool(hadoop('command -v python3 || true').strip())

if has_python:
    mapper_file, reducer_file = 'mapper.py', 'reducer.py'
    mapper_cmd, reducer_cmd = 'python3 mapper.py', 'python3 reducer.py'
    print('Using Python mapper/reducer ✓')
else:
    mapper_file, reducer_file = 'mapper.awk', 'reducer.awk'
    mapper_cmd, reducer_cmd = 'awk -f mapper.awk', 'awk -f reducer.awk'
    print('Python not found in image — falling back to awk ✓')

STAGE = '/tmp/stream'
for fname in (mapper_file, reducer_file):
    docker_cp(f'../temp/weather/{fname}', f'{STAGE}/{fname}')
print(f'Staged {mapper_file}, {reducer_file} → namenode:{STAGE}')

## 5. Submit the Streaming job on YARN

### CLI equivalent
```bash
yarn jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -files mapper.py,reducer.py \
    -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
    -input /datasets/weather -output /output/weather
```

Open the [ResourceManager UI (localhost:8088)](http://localhost:8088), then run this cell to watch the job execute.

In [ ]:
STREAMING_JAR = hadoop(
    'ls $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar'
).strip().splitlines()[-1]
print('Streaming jar:', STREAMING_JAR)

OUTPUT_DIR = '/output/weather'
try:
    client.delete(OUTPUT_DIR, recursive=True)
except Exception:
    pass

job = (f'cd {STAGE} && yarn jar {STREAMING_JAR} '
       f'-files {mapper_file},{reducer_file} '
       f'-mapper "{mapper_cmd}" -reducer "{reducer_cmd}" '
       f'-input {INPUT_DIR} -output {OUTPUT_DIR}')

log = hadoop(job)
print(log[-2500:])

## 6. Read the aggregated results

Output columns: `year`, `max_°C`, `mean_°C`, `observations`.

In [ ]:
import io
import pandas as pd

parts = [f for f in client.list(OUTPUT_DIR) if f.startswith('part-')]
frames = []
for part in parts:
    with client.read(f'{OUTPUT_DIR}/{part}') as reader:
        frames.append(pd.read_csv(io.BytesIO(reader.read()), sep='\t',
                                  names=['year', 'max_C', 'mean_C', 'observations']))

weather = pd.concat(frames, ignore_index=True).sort_values('year')
weather

## 7. Cleanup

In [ ]:
import shutil

client.delete(INPUT_DIR, recursive=True)
client.delete(OUTPUT_DIR, recursive=True)
hadoop(f'rm -rf {STAGE}')
shutil.rmtree('../temp/weather', ignore_errors=True)
print('Cleanup complete ✓')

## Summary

| Step | What it demonstrated |
|---|---|
| `.gz` ingest to HDFS | Hadoop reads compressed input transparently |
| Custom mapper/reducer | **your** logic runs as a distributed MapReduce job |
| `-files` + Streaming jar | ship code to the cluster, no Java required |
| ResourceManager UI (8088) | live job, `SUCCEEDED` state |
| Read results back | year → max/mean temperature, consumed with Pandas |

You've now used the cluster for **storage at volume** (lab 04) and **distributed processing** with both a prebuilt jar (lab 05) and your own code (this lab) — the full Hadoop story the docs describe.